In [1]:
import pandas as pd
TRAIN_PATH = 'INPUT/train.csv'
TEST_PATH  = 'INPUT/test.csv'
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

In [8]:
import spacy

nlp = spacy.load("en_core_web_sm")
def features(text):
    doc = nlp(text)
    return len([token for token in doc])
#    return len([token for token in doc if token.is_alpha])

df = df_train
df['word_count'] = df['body'].apply(features)

In [10]:
total_words = df['body'].apply(features).sum()
print("Total words:", total_words)

Total words: 70569


In [49]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 6.4 MB/s  0:00:00


In [64]:
import re
import emoji
import spacy

nlp = spacy.load("en_core_web_sm")

def additional_features(text):
    one_line_text = text.replace('\n', ' ')
    one_line_text = re.sub(r'\s+', ' ', one_line_text).strip()
    
    doc = nlp(one_line_text)
    http_urls = []
    https_urls = []

    for token in doc:
        if token.like_url:
            if token.text.startswith('http://'):
                http_urls.append(token.text)
            elif token.text.startswith('https://'):
                https_urls.append(token.text)

    # Phone number regex (simplified)
    phone_pattern = r'(\+?\d{1,3}[-.\s]?)?(\(?\d{1,4}\)?[-.\s]?)?[\d\s.-]{5,}'
    phone_matches = re.findall(phone_pattern, one_line_text)

    # Count emojis (character-level)
    emoji_count = sum(1 for char in one_line_text if emoji.is_emoji(char))

    lemmas = [token.lemma_ for token in doc if not token.is_punct]
    lemmatized_comment = " ".join(lemmas)
    
    features = {
        'comment': lemmatized_comment,
        'body_length': sum(len(token.text) for token in doc if not token.is_space),
        'word_counts': len([token.text for token in doc]),
        'alpha_word_counts': len([token.text for token in doc if token.is_alpha]),
        'email_counts': len([token.text for token in doc if token.like_email]),
        'phone_counts': len(phone_matches),
        'http_counts': len(http_urls),
        'https_counts': len(https_urls),
        'emoji_counts': emoji_count,
    }
    

    return features


In [65]:
# Apply the function to each row
features_df = df['body'].apply(additional_features).apply(pd.Series)

# Merge the features back into the original DataFrame
df = pd.concat([df, features_df], axis=1)

In [ ]:
df.head(10)

In [60]:
text = """#Rapper \n🚨Straight Outta Cross Keys SC 🚨YouTu...
hhhh https://en.wikipedia.org/w...  xxx@gmail.com 788888
gggggg
"""
f=additional_features(text)
print (f)

doc = nlp(text)
for token in doc:
    print(f"{token.text} -> {token.lemma_}")

{'body_length': 99, 'word_counts': 17, 'alpha_word_counts': 9, 'email_counts': 1, 'phone_counts': 1, 'http_counts': 0, 'https_counts': 1, 'emoji_counts': 2}
# -> #
Rapper -> Rapper

 -> 

🚨 -> 🚨
Straight -> Straight
Outta -> Outta
Cross -> Cross
Keys -> Keys
SC -> SC
🚨 -> 🚨
YouTu -> YouTu
... -> ...

 -> 

hhhh -> hhhh
https://en.wikipedia.org/w -> https://en.wikipedia.org/w
... -> ...
  ->  
xxx@gmail.com -> xxx@gmail.com
788888 -> 788888

 -> 

gggggg -> gggggg

 -> 



In [41]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp("Banks don't want you to know this! Click here ")

for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

In [47]:
from collections import Counter
def get_word_frequencies(text):
    doc = nlp(text.lower())
    tokens = [token.text for token in doc if token.is_alpha and not token.is_stop]
    return Counter(tokens)

# Apply to the DataFrame column and combine
all_freq = Counter()
df['body'].apply(lambda x: all_freq.update(get_word_frequencies(x)))

# Convert to DataFrame if needed
freq_df = pd.DataFrame(all_freq.items(), columns=['word', 'frequency']).sort_values(by='frequency', ascending=False)

print(freq_df)

                    word  frequency
5                 stream        150
232                   hd        149
139                 like        149
18                  free        146
1                   want        121
...                  ...        ...
3571  şeytanauymahayırde          1
3570              dilmen          1
3569              çağdaş          1
3568       corlutravesti          1
3555                  ek          1

[6492 rows x 2 columns]


In [48]:
freq_df.to_csv('xxx.csv')

In [12]:
df.head()

,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation,word_count
0,0,Banks don't want you to know this! Click here ...,"No Advertising: Spam, referral links, unsolici...",Futurology,If you could tell your younger self something ...,hunt for lady for jack off in neighbourhood ht...,Watch Golden Globe Awards 2017 Live Online in ...,"DOUBLE CEE x BANDS EPPS - ""BIRDS""\n\nDOWNLOAD/...",0,15
1,1,SD Stream [ ENG Link 1] (http://www.sportsstre...,"No Advertising: Spam, referral links, unsolici...",soccerstreams,[I wanna kiss you all over! Stunning!](http://...,LOLGA.COM is One of the First Professional Onl...,#Rapper \n🚨Straight Outta Cross Keys SC 🚨YouTu...,[15 Amazing Hidden Features Of Google Search Y...,0,10
2,2,Lol. Try appealing the ban and say you won't d...,No legal advice: Do not offer or request legal...,pcmasterrace,Don't break up with him or call the cops. If ...,It'll be dismissed: https://en.wikipedia.org/w...,Where is there a site that still works where y...,Because this statement of his is true. It isn'...,1,15
3,3,she will come your home open her legs with an...,"No Advertising: Spam, referral links, unsolici...",sex,Selling Tyrande codes for 3€ to paypal. PM. \n...,tight pussy watch for your cock get her at thi...,NSFW(obviously) http://spankbang.com/iy3u/vide...,Good News ::Download WhatsApp 2.16.230 APK for...,1,14
4,4,code free tyrande --->>> [Imgur](http://i.imgu...,"No Advertising: Spam, referral links, unsolici...",hearthstone,wow!! amazing reminds me of the old days.Well...,seek for lady for sex in around http://p77.pl/...,must be watch movie https://sites.google.com/s...,We're streaming Pokemon Veitnamese Crystal RIG...,1,36
